In [2]:
#import package
import pandas as pd
import numpy as np
import os
import time
import warnings

#Export/ import model
import joblib

#Predifined function
from src.dataset import loader, data_info
from src.eda import data_profiling, visualizer_all_metrics, find_optimal_eps
from src.dictionaries import dictionary
from src.pre_processing import smote, tomek_links, adasyn

#Imputer
from sklearn.impute import KNNImputer

#Scaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

#Dimension reduction
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

#Noise detection model
from sklearn.cluster import DBSCAN
from collections import Counter

#Clustering model
from sklearn.cluster import (KMeans, SpectralClustering, AgglomerativeClustering)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

#Prediction model
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

#Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Asking window
import tkinter as tk
from tkinter import messagebox
from sklearn.exceptions import UndefinedMetricWarning

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings('ignore', message='n_jobs value 1 overridden')

def asking_window(title, message):
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    response = messagebox.askyesno(title, message)
    root.destroy()
    return response

warnings.filterwarnings('ignore', message='n_jobs value 1 overridden')

output_profiling = "../resources/profiling"

output_clustering_scaler= "../resources/model/clustering/scaler"
output_clustering_model= "../resources/model/clustering/model"
output_clustering_result = "../resources/model/clustering/result"

output_predicting_scaler= "../resources/model/predict/scaler"
output_predicting_model= "../resources/model/predict/model"
output_predicting_result= "../resources/model/predict/result"


def plot_clustering_results(viz_df, labels, title):
    """
    Vẽ 3 subplot PCA, t-SNE, UMAP tô màu theo nhãn cụm vừa tìm được.
    """
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))

    # Thiết lập màu sắc: -1 (nhiễu) sẽ có màu xám, các cụm còn lại dùng bảng màu Spectral
    unique_labels = set(labels)
    palette = sns.color_palette("Spectral", n_colors=len(unique_labels))
    if -1 in unique_labels:
        palette = {lbl: palette[i] if lbl != -1 else (0.8, 0.8, 0.8) for i, lbl in enumerate(unique_labels)}

    #PCA
    sns.scatterplot(
        ax=axes[0], x='PCA1', y='PCA2', data=viz_df,
        hue=labels, palette=palette, alpha=0.4, s=10, legend=None
    )
    axes[0].set_title('PCA - Tổng quan phương sai', fontsize=14)
    axes[0].grid(True, linestyle='--', alpha=0.4)

    #t-SNE
    sns.scatterplot(
        ax=axes[1], x='TSNE1', y='TSNE2', data=viz_df,
        hue=labels, palette=palette, alpha=0.4, s=10, legend=None
    )
    axes[1].set_title('t-SNE - Cụm mật độ cục bộ', fontsize=14)
    axes[1].grid(True, linestyle='--', alpha=0.4)

    #UMAP
    sns.scatterplot(
        ax=axes[2], x='UMAP1', y='UMAP2', data=viz_df,
        hue=labels, palette=palette, alpha=0.4, s=10, legend=None
    )
    axes[2].set_title('UMAP - Cấu trúc toàn cục & cục bộ', fontsize=14)
    axes[2].grid(True, linestyle='--', alpha=0.4)

    # Tiêu đề tổng hợp
    plt.suptitle(f'{title}\nSo sánh các phương pháp trực quan hóa cấu trúc dữ liệu', fontsize=16, y=1.05,
                 fontweight='bold')

    plt.tight_layout()
    plt.show()

def run_clustering_benchmark_with_viz(data, k_list=[2, 3, 4]):

    print("Chuẩn bị không gian trực quan (PCA, t-SNE, UMAP)...")
    pca_res = PCA(n_components=2).fit_transform(data)
    tsne_res = TSNE(n_components=2, random_state=42).fit_transform(data)
    umap_res = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(data)

    viz_df = pd.DataFrame({
        'PCA1': pca_res[:, 0], 'PCA2': pca_res[:, 1],
        'TSNE1': tsne_res[:, 0], 'TSNE2': tsne_res[:, 1],
        'UMAP1': umap_res[:, 0], 'UMAP2': umap_res[:, 1]
    })

    results = []

    def execute_and_log(name, model, params):
        start_time = time.time()
        try:
            labels = model.fit_predict(data)
            end_time = time.time()
            runtime = end_time - start_time

            counts = Counter(labels)
            n_noise = counts.pop(-1, 0)
            n_clusters = len(counts)
            cluster_sizes = {f"C{k}": v for k, v in sorted(counts.items())}

            if n_clusters > 1:
                mask = labels != -1
                s_score = silhouette_score(data[mask], labels[mask])
                dbi_score = davies_bouldin_score(data[mask], labels[mask])

                results.append({
                    "Algorithm": name, "Parameters": params, "n_clusters": n_clusters,
                    "Cluster_Sizes": cluster_sizes, "Noise_Count": n_noise,
                    "Silhouette": round(s_score, 4), "DBI": round(dbi_score, 4),
                    "Runtime (s)": round(runtime, 4)
                })

                title = f"{name} | Params: {params} | Sil: {round(s_score, 3)} | DBI: {round(dbi_score, 4)} | Noise: {n_noise}"
                plot_clustering_results(viz_df, labels, title)
                print(f"Hoàn thành {name} | Clusters: {n_clusters}")

        except Exception as e:
            print(f"Lỗi {name} ({params}): {e}")

    for k in k_list:
        models_k = {
            "K-Means": KMeans(n_clusters=k, random_state=42, n_init=10),
            "GMM": GaussianMixture(n_components=k, random_state=42),
            "Agglomerative": AgglomerativeClustering(n_clusters=k),
            "Spectral_NN_20": SpectralClustering(
            n_clusters=k, affinity='nearest_neighbors', n_neighbors=20, random_state=42),
            "Spectral_NN_50": SpectralClustering(
                n_clusters=k, affinity='nearest_neighbors', n_neighbors=50, random_state=42),
            "Spectral_RBF_high": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=1.0, random_state=42),
            "Spectral_RBF_mid": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=0.1, random_state=42),
            "Spectral_RBF_low": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=0.01, random_state=42),
            "Spectral_Laplacian": SpectralClustering(
                n_clusters=k, affinity='laplacian', random_state=42)
            }
        for name, model in models_k.items():
            execute_and_log(name, model, {"k": k})
    return pd.DataFrame(results)

def train_multi_model(data_name, X, y):
    """
    Hàm huấn luyện tất cả các mô hình trên từng biến thể dữ liệu.
    data_variants: Dictionary dạng {"Tên bộ dữ liệu": (X, y)}
    """

    models = {
        "Gradient-Boosted": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42),
        "Deep Learning": MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.01, max_iter=1000, random_state=42),
        "Naïve Bayes": GaussianNB(),
        "SGD Classifier": SGDClassifier(loss='hinge',class_weight='balanced', random_state=42)
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision_weighted',
        'recall': 'recall_weighted',
        'f1': 'f1_weighted'
    }

    final_results = []

    print(f"{'='*60}")
    print(f"BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN")
    print(f"{'='*60}")

    print(f"\n>>> Thông tin bộ dữ liệu: {data_name} | Kích thước: {X.shape}")

    for model_name, model in models.items():
        try:
            print(model_name)
            start_t = time.perf_counter()
            # Tính toán cross-validation
            cv_results = cross_validate(
                model, X, y,
                cv=cv,
                scoring=scoring_metrics,
                error_score='raise',
                n_jobs=-1
            )
            end_t = time.perf_counter()
            total_time = end_t - start_t

            # Lưu kết quả
            r = {
                "Data Variant": data_name,
                "Model": model_name,
                "Accuracy": np.mean(cv_results['test_accuracy']),
                "Std Dev": np.std(cv_results['test_accuracy']),
                "Precision": np.mean(cv_results['test_precision']),
                "Recall": np.mean(cv_results['test_recall']),
                "F1-Score": np.mean(cv_results['test_f1']),
                "Time (s)": np.mean(cv_results['fit_time'])
            }

            final_results.append(r)
            print(f"  + {model_name.ljust(20)}: Hoàn tất (Accuracy: {r['Accuracy']:.4f} | F1: {r['F1-Score']:.4f} | Time : {total_time} )")

        except Exception as e:
            print(f"  + {model_name.ljust(20)}: LỖI - {str(e)[:50]}...")

    #XUẤT KẾT QUẢ
    df_final = pd.DataFrame(final_results)

    # Định dạng bảng
    df_display = df_final.copy()
    for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
        df_display[col] = df_display[col].apply(lambda x: f"{x*100:.2f}%")

    # Thêm cột Std Dev vào Accuracy
    df_display['Accuracy'] = df_display['Accuracy'] + " (±" + (df_final['Std Dev']*100).map("{:.2f}%".format) + ")"
    df_display = df_display.drop(columns=['Std Dev'])

    # Sắp xếp theo F1-Score hoặc Accuracy
    df_display = df_display.sort_values(by=['Data Variant', 'F1-Score'], ascending=[True, False])

    print("\n" + "="*80)
    print("BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ")
    print("="*80)
    print(df_display.to_string(index=False))
    return df_final

def split_data_by_clusters(df, cluster_col):
    """
    Tách DataFrame gốc thành dictionary các DataFrame con theo cluster.
    """
    unique_clusters = sorted(df[cluster_col].unique())
    cluster_dict = {}
    for cid in unique_clusters:
        name = f"cluster_{cid}" if cid != -1 else "cluster_noise"
        cluster_dict[name] = df[df[cluster_col] == cid].copy()
        print(f"  + {name.ljust(15)}: {len(cluster_dict[name])} dòng")
    return cluster_dict

def preprocess_clusters(cluster_dict, target_col):
    """
    Lặp qua từng cluster để:
    1. Scale dữ liệu
    2. Lọc lớp hiếm
    3. Tạo các biến thể Resampling (SMOTE, ADASYN,...)
    """

    processed_results = {}

    print(f"\n{'='*50}")
    print(f"BƯỚC 2: SCALING & RESAMPLING CHI TIẾT")

    for cluster_name, df_raw in cluster_dict.items():
        if len(df_raw) < 10:
            print(f"  ! {cluster_name}: Bỏ qua (quá ít dữ liệu)")
            continue

        print(f"  > Đang xử lý {cluster_name}...")
        # Nhóm dùng RobustScaler
        robust_features = ['BAS_werkzaamheid_resp', 'afg_kinderen_huishouden']

        # Nhóm dùng MinMaxScaler
        minmax_features = ['GenderID', 'SPSS_Regio5', 'step2', 'step3']

        # Nhóm dùng StandardScaler
        standard_features = ['Age']

        target = 'step1'

        # XÂY DỰNG BỘ PREPROCESSOR
        preprocessor_cluster = ColumnTransformer(
            transformers=[
                ('rob', RobustScaler(), robust_features),
                ('mms', MinMaxScaler(), minmax_features),
                ('std', StandardScaler(), standard_features)
            ],
            remainder='drop'
        )

        X_scaled = preprocessor_cluster.fit_transform(df_raw)

        s2_predicting_preprocessor_file = os.path.join(output_predicting_scaler, f's2_predicting_preprocessor_{cluster_name}.pkl')
        joblib.dump(preprocessor_cluster, s2_predicting_preprocessor_file)

        y = df_raw[target_col].values

        min_samples = 6
        counts = Counter(y)
        valid_classes = [cls for cls, count in counts.items() if count >= min_samples]

        if len(valid_classes) < 2:
            print(f"    - {cluster_name}: Không đủ lớp mục tiêu để Resampling.")
            continue

        mask = np.isin(y, valid_classes)
        X_f = X_scaled[mask]
        y_f = y[mask]

        #Resampling
        try:
            X_sm, y_sm = smote(X_f, y_f)
            X_tm, y_tm = tomek_links(X_f, y_f)
            X_st, y_st = tomek_links(X_sm, y_sm)
            X_ad, y_ad = adasyn(X_f, y_f)

            # Lưu vào dictionary
            processed_results[cluster_name] = {
                "Original": (X_f, y_f),
                "SMOTE": (X_sm, y_sm),
                "Tomek": (X_tm, y_tm),
                "SMOTE-Tomek": (X_st, y_st),
                "ADASYN": (X_ad, y_ad)
            }
            print(f"{cluster_name}: Hoàn tất tạo 5 biến thể dữ liệu.")

        except Exception as e:
            print(f"{cluster_name}: Lỗi Resampling - {str(e)}")

    return processed_results

TravelDataJourneys_path = r'D:\MASTER\2.1. ADM\FINAL_PROJECT\CTMJ_ds\A_TravelDataJourneys.csv'
TravelDataUsers_path = r'D:\MASTER\2.1. ADM\FINAL_PROJECT\CTMJ_ds\B_TravelDataUsers.csv'
trigger_api = asking_window('Ask for crawling data', 'Do you want to get api data?')
if trigger_api:
    #Get data
    print("Getting data from API")
    article_id = "23690811"
    api_url = f"https://api.figshare.com/v2/articles/{article_id}"
    dataset = loader.get_data(api_url)
    data_info.dataset_info(dataset)
    TravelDataUsers = dataset["B_TravelDataUsers.csv"]
    TravelDataJourneys = dataset["A_TravelDataJourneys.csv"]
else:
    try:
        TravelDataJourneys = pd.read_csv(TravelDataJourneys_path)
        TravelDataUsers = pd.read_csv(TravelDataUsers_path)
    except:
        print("Can not get data files. Back to get data from API")
        print("Getting data from API")
        article_id = "23690811"
        api_url = f"https://api.figshare.com/v2/articles/{article_id}"
        dataset = loader.get_data(api_url)
        data_info.dataset_info(dataset)
        TravelDataUsers = dataset["B_TravelDataUsers.csv"]
        TravelDataJourneys = dataset["A_TravelDataJourneys.csv"]

print('All dataset are ready for use')

All dataset are ready for use


In [3]:
#Export - có thể không run
trigger_profiling = asking_window("Ask for profiling", "Do you want to get profiling data?")

if trigger_profiling:
    # User Dashboard information
    user_output_filename = "TravelDataUsers_data_profiling.html"
    user_dashboard_title = "Travel Users Dataset Profiling"

    #Run
    data_profiling(TravelDataUsers, output_profiling ,user_output_filename, user_dashboard_title)
    print('Done profiling for TravelDataUsers')

    #Journey Dashboard information
    journey_output_filename = "TravelDataJourneys_data_profiling.html"
    journey_dashboard_title = "Travel Journeys Dataset Profiling"

    #Run
    data_profiling(TravelDataJourneys, output_profiling,journey_output_filename, journey_dashboard_title)
    print('Done profiling for TravelDataJourneys')
else:
    print('Nothing to do!')

Running Profile Report...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:00<00:00, 12061.26it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...


100%|██████████| 10/10 [00:40<00:00,  4.04s/it][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Open report: D:\MASTER\2.1. ADM\ctmj\src\resources\profiling\TravelDataJourneys_data_profiling.html
Done profiling for TravelDataJourneys


In [4]:
#Preparing data for Clustering
print('Preparing data for Clustering')
dictionary = dictionary()
touchpoint_mapping = dictionary.type_touch

#Preparing data for Clustering
user_data = TravelDataUsers[~TravelDataUsers['SPSS_Regio5'].isnull()]
selected_column = [
    'UserID',
    'GenderID', #gioi tinh
    'Age', #tuoi
    'SPSS_Regio5', #vung
    'BAS_huishoudgrootte', #quy mo ho gia dinh (so nguoi trong nha)
    'BAS_werkzaamheid_resp', #nghe nghiep
    'BAS_bruto_jaarinkomen', #tong thu nhap truoc thue
    'afg_kinderen_huishouden', #so luong tre em trong gia dinh
    'AFG_sk2015', #tang lop xa hoi
    'BAS_voltooide_opleiding8_resp', #trinh do hoc van
    'SPSS_Lifestage' #giai doan cuoc doi
]

user_data = user_data[selected_column]

#Xử lý dữ liệu thiếu
imputer = KNNImputer(n_neighbors=5)
user_data_imputed_array = imputer.fit_transform(user_data)

user_data = pd.DataFrame(
    user_data_imputed_array,
    columns=user_data.columns,
    index=user_data.index
)
trigger_profiling = asking_window("Ask for profiling", "Do you want to get processed user profiling data?")
if trigger_profiling:
    processed_user_output_filename = "Processed_user_data_profiling.html"
    processed_user_dashboard_title = "Processed Travel Users Dataset Profiling"

    #Run
    data_profiling(user_data, output_profiling,processed_user_output_filename, processed_user_dashboard_title)
else:
    print('Nothing to do!')

Preparing data for Clustering
Running Profile Report...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 11/11 [00:00<00:00, 146.72it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Open report: D:\MASTER\2.1. ADM\ctmj\src\resources\profiling\Processed_user_data_profiling.html


In [5]:

#user_data info
print(user_data.info())
print(user_data.head())

#Preparing data for Prediction
print('Preparing data for Prediction')
#Dictionary mapping
journey_data = TravelDataJourneys[TravelDataJourneys['Duration'] > 0].copy()
#Duration > 0
journey_data['TIMESPSS'] = pd.to_datetime(journey_data['TIMESPSS'])
journey_data = journey_data.sort_values(by=['UserID', 'TIMESPSS']).reset_index(drop=True)

#Đánh dấu các click liên tiếp cùng loại của cùng 1 User
change_mask = (journey_data['type_touch'] != journey_data['type_touch'].shift()) | (journey_data['UserID'] != journey_data['UserID'].shift())
journey_data['group_id'] = change_mask.cumsum()

#Aggregation
journey_data_compressed = journey_data.groupby(['UserID', 'group_id', 'type_touch']).agg({
    'TIMESPSS': 'first',
    'Duration': 'sum',
    'PurchaseID': 'first',
    'DEVICE_TYPE': 'first',
    'purchase_own': 'max',
    'purchase_any': 'max',
    'MobilePanel': 'first',
    'FixedPanel': 'first'
}).reset_index()

#Mapping tên điểm chạm
journey_data_compressed['touchpoint_name'] = journey_data_compressed['type_touch'].map(touchpoint_mapping)

user_id_list = user_data['UserID'].unique()
journey_data_compressed = journey_data_compressed[journey_data_compressed['UserID'].isin(user_id_list)]

#Dọn dẹp và kiểm tra
journey_data_compressed = journey_data_compressed.drop(columns='group_id')

print(f"Số dòng sau khi nén: {len(journey_data_compressed)}")

#journey_data info
print(journey_data_compressed.info())
print(journey_data_compressed.head())
#Export journey data - có thể không chạy
trigger_export_data = asking_window("Ask for exporting data",
                                    "Do you want to get processed journey data?")
if trigger_export_data:
    journey_data_compressed[['UserID', 'touchpoint_name', 'TIMESPSS']].to_csv(f"{output_predicting_result}/cjm.csv", index=False)
    print(f'Journey data has been exported to {output_predicting_result}/cjm.csv')
else:
    print('Nothing to do!')

#Export journey data profiling
trigger_profiling = asking_window("Ask for profiling data",
                                 "Do you want to get processed journey profiling data?")
if trigger_profiling:
    processed_journey_output_filename = "Processed_journey_data_profiling.html"
    processed_journey_dashboard_title = "Processed Travel Journey Dataset Profiling"

    # Run
    data_profiling(journey_data_compressed, output_profiling, processed_journey_output_filename, processed_journey_dashboard_title)
else:
    print('Nothing to do!')

<class 'pandas.core.frame.DataFrame'>
Index: 8075 entries, 2 to 9677
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   UserID                         8075 non-null   float64
 1   GenderID                       8075 non-null   float64
 2   Age                            8075 non-null   float64
 3   SPSS_Regio5                    8075 non-null   float64
 4   BAS_huishoudgrootte            8075 non-null   float64
 5   BAS_werkzaamheid_resp          8075 non-null   float64
 6   BAS_bruto_jaarinkomen          8075 non-null   float64
 7   afg_kinderen_huishouden        8075 non-null   float64
 8   AFG_sk2015                     8075 non-null   float64
 9   BAS_voltooide_opleiding8_resp  8075 non-null   float64
 10  SPSS_Lifestage                 8075 non-null   float64
dtypes: float64(11)
memory usage: 757.0 KB
None
   UserID  GenderID   Age  SPSS_Regio5  BAS_huishoudgrootte  \
2     1

OSError: Cannot save file into a non-existent directory: '..\resources\model\predict\result'